In [0]:
from pyspark.sql.functions import col

df_oc = spark.table(
    "chilecompra.silver.ordenes_compra"
)

df_licitaciones = spark.table(
    "chilecompra.silver.licitaciones"
)

In [0]:
df_oc_licitaciones = (
    df_oc.alias("oc")
    .join(
        df_licitaciones.alias("l"),
        col("oc.codigo_licitacion") == col("l.codigo_externo"),
        how="left"
    )
)

In [0]:
from pyspark.sql.functions import (
    when,
    year,
    month,
    countDistinct,
    sum
)

df_oc_licitaciones_metricas = (
    df_oc_licitaciones
    .filter(col("oc.fecha_envio").isNotNull())

    .withColumn(
        "tiene_licitacion",
        when(
            col("oc.codigo_licitacion").isNotNull(),
            1
        ).otherwise(0)
    )

    .withColumn(
        "tiene_match_api",
        when(
            col("l.codigo_externo").isNotNull(),
            1
        ).otherwise(0)
    )

    .groupBy(
        year(col("oc.fecha_envio")).alias("anio"),
        month(col("oc.fecha_envio")).alias("mes")
    )

    .agg(
        countDistinct(col("oc.codigo")).alias("total_ordenes"),
        sum("tiene_licitacion").alias("ordenes_con_licitacion"),
        sum("tiene_match_api").alias("ordenes_con_match_api")
    )

    .withColumn(
        "ordenes_sin_licitacion",
        col("total_ordenes") - col("ordenes_con_licitacion")
    )

    .withColumn(
        "porcentaje_con_licitacion",
        col("ordenes_con_licitacion")
        / col("total_ordenes") * 100
    )

    .orderBy("anio", "mes")
)

In [0]:
total_rows = df_oc_licitaciones_metricas.count()

null_keys = (
    df_oc_licitaciones_metricas
    .filter(
        col("anio").isNull()
        | col("mes").isNull()
    )
    .count()
)

distinct_keys = (
    df_oc_licitaciones_metricas
    .select("anio", "mes")
    .distinct()
    .count()
)

invalid_counts = (
    df_oc_licitaciones_metricas
    .filter(
        (col("total_ordenes") <= 0)
        | (col("ordenes_con_licitacion") < 0)
        | (col("ordenes_sin_licitacion") < 0)
        | (col("ordenes_con_match_api") < 0)
    )
    .count()
)

invalid_balance = (
    df_oc_licitaciones_metricas
    .filter(
        col("total_ordenes")
        != col("ordenes_con_licitacion") + col("ordenes_sin_licitacion")
    )
    .count()
)

invalid_percentage = (
    df_oc_licitaciones_metricas
    .filter(
        (col("porcentaje_con_licitacion") < 0)
        | (col("porcentaje_con_licitacion") > 100)
    )
    .count()
)

if total_rows == 0:
    raise ValueError("DQ FAILED: OC-licitacion aggregation produced 0 rows")

if null_keys > 0:
    raise ValueError(
        f"DQ FAILED: {null_keys} rows have NULL year/month"
    )

if total_rows != distinct_keys:
    raise ValueError(
        "DQ FAILED: duplicated year/month keys"
    )

if invalid_counts > 0:
    raise ValueError(
        f"DQ FAILED: {invalid_counts} rows have invalid counts"
    )

if invalid_balance > 0:
    raise ValueError(
        f"DQ FAILED: {invalid_balance} rows do not reconcile total orders"
    )

if invalid_percentage > 0:
    raise ValueError(
        f"DQ FAILED: {invalid_percentage} rows have invalid percentages"
    )

print(
    f"Pre-write OC-licitaciones Gold DQ passed: {total_rows} months"
)

In [0]:
target_table = "chilecompra.gold.oc_licitaciones_mensual"

(
    df_oc_licitaciones_metricas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
df_gold_oc_licitaciones = spark.table(target_table)

gold_rows = df_gold_oc_licitaciones.count()

gold_distinct_keys = (
    df_gold_oc_licitaciones
    .select("anio", "mes")
    .distinct()
    .count()
)

gold_invalid_balance = (
    df_gold_oc_licitaciones
    .filter(
        col("total_ordenes")
        != col("ordenes_con_licitacion") + col("ordenes_sin_licitacion")
    )
    .count()
)

if gold_rows != total_rows:
    raise ValueError(
        "DQ FAILED: Gold row count does not match source aggregation"
    )

if gold_rows != gold_distinct_keys:
    raise ValueError(
        "DQ FAILED: duplicated year/month keys in Gold"
    )

if gold_invalid_balance > 0:
    raise ValueError(
        f"DQ FAILED: {gold_invalid_balance} Gold rows do not reconcile"
    )

print(
    f"Gold oc_licitaciones_mensual DQ passed: {gold_rows} months"
)